# Step 1: data and linear probes

One-time setup:
1. Make a folder `grsats` in My Drive and upload `HAM10000_images_part_1.zip`, `HAM10000_images_part_2.zip`, and `finalfitz17k.zip` (the `finalfitz17k` folder zipped) into it.
2. Make a GitHub token with read access to the repo and add it in Colab under Secrets (key icon) as `GH_TOKEN`.
3. Runtime > Change runtime type > T4 GPU.

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")

token = userdata.get("GH_TOKEN")
!git clone -q https://{token}@github.com/adipatel11/gr-sats-term-paper.git /content/repo || git -C /content/repo pull -q
%cd /content/repo/code
!pip install -q -r requirements.txt

# outputs go to drive so they survive a disconnect
os.environ["GRSATS_OUT"] = "/content/drive/MyDrive/grsats/outputs"

Mounted at /content/drive
/content/repo/code


In [2]:
# unzip images to the colab disk, reading thousands of files straight from drive is really slow
# (same folders you get on a mac by double clicking the zips)
ham = "../data/HAM10000/dataverse_files"
!mkdir -p {ham}
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_1.zip -d {ham}/HAM10000_images_part_1
!unzip -qn /content/drive/MyDrive/grsats/HAM10000_images_part_2.zip -d {ham}/HAM10000_images_part_2
!unzip -qn /content/drive/MyDrive/grsats/finalfitz17k.zip -d ../data

# counts should match the paper and missing image files should be 0
!python -m scripts.check_data
!python -m pytest -q tests


ham: 9,948 images
  train        8,164 (1,740, 21.3%)
  val          568 (63, 11.1%)
  test         1,216 (149, 12.3%)
  pool         1,784 (212, 11.9%)
  pool female  820 (80, 9.8%)
  pool male    964 (132, 13.7%)
  pool 60+     578 (150, 26.0%)
  pool <60     1,206 (62, 5.1%)
  missing image files: 9948

fitz: 11,098 images
  train        7,779 (1,144, 14.7%)
  val          1,100 (154, 14.0%)
  test         2,219 (325, 14.6%)
  pool         3,319 (479, 14.4%)
  pool I-II    1,794 (287, 16.0%)
  pool III-IV  1,163 (160, 13.8%)
  pool V-VI    362 (32, 8.8%)
  missing image files: 0
.......                                                                  [100%]
7 passed in 1.77s


In [3]:
for ds in ["ham", "fitz"]:
    for bb in ["dinov2", "biomedclip"]:
        !python -m scripts.extract_features --dataset {ds} --backbone {bb}

9948 ham images not found, check the unzip step
9948 ham images not found, check the unzip step
fitz_dinov2_features.npz already done, skipping
fitz_biomedclip_features.npz already done, skipping


In [4]:
# cpu only, could also run this locally after downloading the outputs folder
for ds in ["ham", "fitz"]:
    for bb in ["dinov2", "biomedclip"]:
        !python -m scripts.train_probes --dataset {ds} --backbone {bb}

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/repo/code/scripts/train_probes.py", line 21, in <module>
    f = np.load(OUT / f"{name}_features.npz")
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 454, in load
    fid = stack.enter_context(open(os.fspath(file), "rb"))
                              ~~~~^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/grsats/outputs/ham_dinov2_features.npz'
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/repo/code/scripts/train_probes.py", line 21, in <module>
    f = np.load(OUT / f"{name}_features.npz")
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 454, in load
    fid = stack.enter_context(open(os.fspath(file), "rb")